# Four-class model improvement

This experiment keeps the existing group-aware fit/validation split. It does not read the official test data. We compare small, regularized Decision Tree, Random Forest, and HistGradientBoosting configurations using grouped cross-validation inside the fit partition, then evaluate only the strongest candidates on the untouched validation partition.

The conventional normal false-alarm rate is: normal samples predicted as any attack divided by all normal samples. Attack recall is the mean recall of Exploits, Fuzzers, and DoS.


In [ ]:
from pathlib import Path
import json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, confusion_matrix
import joblib

EXPERIMENT = Path.cwd()
if (EXPERIMENT / 'aftab' / 'data' / 'processed').exists():
    PROJECT = EXPERIMENT
    EXPERIMENT = PROJECT / 'aftab' / 'experiments' / 'four_class' / 'improvement'
else:
    PROJECT = EXPERIMENT.parents[3]
DATA = PROJECT / 'aftab' / 'data' / 'processed'
RESULTS = EXPERIMENT / 'results'; PLOTS = RESULTS / 'plots'; MODELS = EXPERIMENT / 'models'
for p in [RESULTS, PLOTS, MODELS]: p.mkdir(parents=True, exist_ok=True)
CLASSES = ['Normal','Exploits','Fuzzers','DoS']; TARGET='attack_cat'
PREDICTORS = ['dur','proto','service','state','spkts','dpkts','sbytes','dbytes','rate','sttl','dttl','sload','dload','sloss','dloss','sinpkt','dinpkt','sjit','djit','swin','stcpb','dtcpb','dwin','tcprtt','synack','ackdat','smean','dmean','trans_depth','response_body_len','ct_srv_src','ct_state_ttl','ct_dst_ltm','ct_src_dport_ltm','ct_dst_sport_ltm','ct_dst_src_ltm','is_ftp_login','ct_ftp_cmd','ct_flw_http_mthd','ct_src_ltm','ct_srv_dst','is_sm_ips_ports']
df = pd.read_csv(DATA/'selected_training.csv')
fit_idx = pd.read_csv(DATA/'fit_indices.csv').iloc[:,0].astype(int).to_numpy()
val_idx = pd.read_csv(DATA/'validation_indices.csv').iloc[:,0].astype(int).to_numpy()
fit4 = df.iloc[fit_idx]; val4 = df.iloc[val_idx]
fit4 = fit4[fit4[TARGET].isin(CLASSES)].copy(); val4 = val4[val4[TARGET].isin(CLASSES)].copy()
print('Fit rows:',len(fit4),'Validation rows:',len(val4))
print('Fit classes:',fit4[TARGET].value_counts().reindex(CLASSES,fill_value=0).to_dict())
print('Validation classes:',val4[TARGET].value_counts().reindex(CLASSES,fill_value=0).to_dict())
assert len(PREDICTORS)==42 and not set(PREDICTORS)&{'id','label','attack_cat'}
def fingerprint(frame):
    return pd.util.hash_pandas_object(frame[PREDICTORS].fillna('__MISSING__').astype(str), index=False).astype(str)
fit_groups=fingerprint(fit4).to_numpy(); val_groups=set(fingerprint(val4))
assert len(set(fit_groups)&val_groups)==0


In [ ]:
numeric = fit4[PREDICTORS].select_dtypes(include=[np.number]).columns.tolist()
categorical = [c for c in PREDICTORS if c not in numeric]
def make_pipe(model, dense=False):
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=not dense, dtype=np.float32)
    prep = ColumnTransformer([('num',Pipeline([('imputer',SimpleImputer(strategy='median'))]),numeric),('cat',Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',encoder)]),categorical)])
    return Pipeline([('preprocess',prep),('model',model)])
X_fit=fit4[PREDICTORS]; y_fit=fit4[TARGET]
cv_sample=fit4.sample(n=min(5000,len(fit4)), random_state=42)
X_cv=cv_sample[PREDICTORS]; y_cv=cv_sample[TARGET]; groups_cv=fingerprint(cv_sample).to_numpy()
X_val=val4[PREDICTORS]; y_val=val4[TARGET]
configs = [
 ('Decision Tree', {'class_weight':None,'max_depth':10,'min_samples_leaf':1}),
 ('Decision Tree', {'class_weight':'balanced','max_depth':20,'min_samples_leaf':10}),
 ('Random Forest', {'class_weight':'balanced_subsample','max_depth':15,'min_samples_leaf':3}),
 ('HistGradientBoosting', {'max_leaf_nodes':31,'learning_rate':0.1,'max_iter':30}),
]
def build(name,p):
    if name=='Decision Tree': model=DecisionTreeClassifier(random_state=42,**p); dense=False
    elif name=='Random Forest': model=RandomForestClassifier(n_estimators=10,n_jobs=1,random_state=42,**p); dense=False
    else: model=HistGradientBoostingClassifier(random_state=42,**p); dense=True
    return make_pipe(model,dense), dense
def metrics(y,pred):
    cm=confusion_matrix(y,pred,labels=CLASSES); prec,rec,f1,sup=precision_recall_fscore_support(y,pred,labels=CLASSES,zero_division=0)
    normal_fpr=cm[1:,0].sum()/cm[1:,:].sum(); return {'accuracy':accuracy_score(y,pred),'macro_f1':f1_score(y,pred,labels=CLASSES,average='macro'),'weighted_f1':f1_score(y,pred,labels=CLASSES,average='weighted'),'normal_false_alarm_rate':normal_fpr,'attack_recall':rec[1:].mean(),'DoS_recall':rec[3],'Fuzzers_recall':rec[2],'Normal_recall':rec[0],'confusion_matrix':cm.tolist()}


In [ ]:
# Grouped cross-validation is used only within the fit partition.
gkf=GroupKFold(n_splits=2); cv_rows=[]
for number,(name,params) in enumerate(configs,1):
    pipe,_=build(name,params); scores=[]; start=time.perf_counter()
    for tr,va in gkf.split(X_cv,y_cv,groups=groups_cv):
        pipe.fit(X_cv.iloc[tr],y_cv.iloc[tr]); scores.append(metrics(y_cv.iloc[va],pipe.predict(X_cv.iloc[va])))
    row={'candidate':number,'model':name,'parameters':json.dumps(params,sort_keys=True),'cv_seconds':time.perf_counter()-start}
    for key in ['accuracy','macro_f1','weighted_f1','normal_false_alarm_rate','attack_recall','DoS_recall','Fuzzers_recall']: row['cv_'+key]=float(np.mean([s[key] for s in scores]))
    cv_rows.append(row); print(number,name,row['cv_macro_f1'])
cv_df=pd.DataFrame(cv_rows).sort_values('cv_macro_f1',ascending=False)
cv_df.to_csv(RESULTS/'cross_validation_results.csv',index=False)
print(cv_df[['candidate','model','cv_macro_f1','cv_normal_false_alarm_rate','cv_DoS_recall','cv_Fuzzers_recall']])


In [ ]:
# Evaluate the best CV candidate from each family on the untouched validation partition.
val_rows=[]; fitted={}
for name in ['Decision Tree','Random Forest','HistGradientBoosting']:
    top=int(cv_df[cv_df.model==name].iloc[0].candidate); cname,cparams=configs[top-1]; pipe,_=build(cname,cparams); start=time.perf_counter(); pipe.fit(X_fit,y_fit); pred=pipe.predict(X_val); m=metrics(y_val,pred); m.update({'candidate':top,'model':cname,'phase':'improvement','parameters':json.dumps(cparams,sort_keys=True),'fit_seconds':time.perf_counter()-start}); val_rows.append(m); fitted[top]=pipe
# Historical baseline is included for context, without rewriting its CSV.
old=pd.read_csv(PROJECT/'aftab/experiments/four_class/results/model_comparison.csv'); base=old[(old.model=='Decision Tree')&(old.phase=='baseline')].iloc[0]
val_rows.append({'candidate':'historical','model':'Existing Decision Tree baseline','phase':'historical','parameters':base.configuration,'fit_seconds':base.fit_seconds,'accuracy':base.accuracy,'macro_f1':base.macro_f1,'weighted_f1':base.weighted_f1,'normal_false_alarm_rate':base.normal_false_positive_rate,'attack_recall':base.attack_recall,'DoS_recall':base.DoS_recall,'Fuzzers_recall':base.Fuzzers_recall,'Normal_recall':base.Normal_recall})
comparison=pd.DataFrame(val_rows); comparison.to_csv(RESULTS/'improvement_comparison.csv',index=False); print(comparison[['model','macro_f1','accuracy','normal_false_alarm_rate','attack_recall','DoS_recall','Fuzzers_recall']])


In [ ]:
# Choose using macro-F1 with the 10% normal false-alarm preference.
new=comparison[comparison.phase=='improvement'].copy(); eligible=new[new.normal_false_alarm_rate<=0.10]; chosen=(eligible if len(eligible) else new).sort_values('macro_f1',ascending=False).iloc[0]
chosen_candidate=int(chosen.candidate); best_pipe=fitted[chosen_candidate]; joblib.dump(best_pipe,MODELS/'best_improved_model.joblib')
fig,ax=plt.subplots(figsize=(8,4)); comparison.set_index('model')[['macro_f1','accuracy','normal_false_alarm_rate','DoS_recall','Fuzzers_recall']].plot(kind='bar',ax=ax); ax.set_ylim(0,1); ax.set_ylabel('Validation score'); fig.tight_layout(); fig.savefig(PLOTS/'improvement_comparison.png',dpi=140); plt.close(fig)
summary=f'''# Four-class improvement summary\n\nPrevious validation macro-F1 was {float(base.macro_f1):.4f}. The selected new candidate is {chosen.model} with parameters {chosen.parameters}.\n\nValidation macro-F1: {chosen.macro_f1:.4f}; accuracy: {chosen.accuracy:.4f}; conventional normal false-alarm rate: {chosen.normal_false_alarm_rate:.4f}; attack recall: {chosen.attack_recall:.4f}; DoS recall: {chosen.DoS_recall:.4f}; Fuzzers recall: {chosen.Fuzzers_recall:.4f}.\n\nThe false-alarm rate is normal samples predicted as any attack divided by all normal samples. Selection used grouped cross-validation inside the fit partition and the untouched validation partition only; the official test data was not read.\n'''
(RESULTS/'improvement_summary.md').write_text(summary)
print('Selected:',chosen.model,chosen.macro_f1)
